In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np

In [2]:
# load German spacy pre-built model
#nlp_de = spacy.load('de_core_news_sm')

In [3]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [4]:
#load csv file
df_male_ct = pd.read_csv("MaleCodedTerms_DE_raw.csv")

In [5]:
#general info
df_male_ct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 7 columns):
MaleCodedWords-German                                                   245 non-null object
Is active                                                               245 non-null int64
Wordtype                                                                245 non-null object
Forms                                                                   198 non-null object
Alternatives-German Input fields job tite | Role | Requirements         185 non-null object
Alternatives-German input fields Company | Company Culture | Goodies    172 non-null object
Subcategory: competitive | Hard-hearted | Rough | Corporate             240 non-null object
dtypes: int64(1), object(6)
memory usage: 13.5+ KB


In [6]:
#first 10 raws
df_male_ct.head(10)

,MaleCodedWords-German,Is active,Wordtype,Forms,Alternatives-German Input fields job tite | Role | Requirements,Alternatives-German input fields Company | Company Culture | Goodies,Subcategory: competitive | Hard-hearted | Rough | Corporate
0,abenteuerlich,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,neugierig | Spass an Neuem | entdeckungsfreudig,experimentierfreudig | wir probieren gerne aus,competitive
1,abenteuerlustig,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,neugierig | Spass an Neuem | entdeckungsfreudig,neugierig | Spass an Neuem | entdeckungsfreudig,competitive
2,abgebrüht,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,ruhig | weise | mit Erfahrung,ruhig | weise | mit Erfahrung,hard-hearted
3,ablehnend,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,du kannst Chance und Risiken gut abwägen,du kannst Chance und Risiken gut abwägen,hard-hearted
4,Aggression,1,s,en,du argumentierst sachlich,du argumentierst sachlich,hard-hearted
5,aggressiv,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,du argumentierst sachlich,du argumentierst sachlich,hard-hearted
6,Aggressivität,1,s,NaN,du argumentierst sachlich,du argumentierst sachlich,hard-hearted
7,aktiv,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,unternehmensfreudig | beweglich | lebendig | d...,unternehmensfreudig | beweglich | lebendig | d...,rough
8,allein,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,seine Kompetenz einsetzen | auf seine Aufgaben...,seine Kompetenz einsetzen | auf seine Aufgaben...,competitive
9,alleinstehend,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,du setzt gerne deine Kompetenz ein | du fokuss...,du setzt gerne deine Kompetenz ein | du fokuss...,competitive


In [7]:
#fill empty values in alternatives with empty string
df_male_ct["Alternatives-German Input fields job tite | Role | Requirements"] = df_male_ct["Alternatives-German Input fields job tite | Role | Requirements"].fillna(value = "")

In [8]:
#check if the filling was correct - did all empty values in this coulumn replaced
df_male_ct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 7 columns):
MaleCodedWords-German                                                   245 non-null object
Is active                                                               245 non-null int64
Wordtype                                                                245 non-null object
Forms                                                                   198 non-null object
Alternatives-German Input fields job tite | Role | Requirements         245 non-null object
Alternatives-German input fields Company | Company Culture | Goodies    172 non-null object
Subcategory: competitive | Hard-hearted | Rough | Corporate             240 non-null object
dtypes: int64(1), object(6)
memory usage: 13.5+ KB


In [9]:
# same for another column of alternatives
df_male_ct["Alternatives-German input fields Company | Company Culture | Goodies"] = df_male_ct["Alternatives-German input fields Company | Company Culture | Goodies"].fillna(value = "")

In [10]:
df_male_ct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 7 columns):
MaleCodedWords-German                                                   245 non-null object
Is active                                                               245 non-null int64
Wordtype                                                                245 non-null object
Forms                                                                   198 non-null object
Alternatives-German Input fields job tite | Role | Requirements         245 non-null object
Alternatives-German input fields Company | Company Culture | Goodies    245 non-null object
Subcategory: competitive | Hard-hearted | Rough | Corporate             240 non-null object
dtypes: int64(1), object(6)
memory usage: 13.5+ KB


In [ ]:
#df_male_ct_new.replace({"":[]}) 

In [11]:
#check how the end files with emty strings looks
df_male_ct.tail()

,MaleCodedWords-German,Is active,Wordtype,Forms,Alternatives-German Input fields job tite | Role | Requirements,Alternatives-German input fields Company | Company Culture | Goodies,Subcategory: competitive | Hard-hearted | Rough | Corporate
240,Wettkämpfe,1,s,NaN,,,competitive
241,zäh,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,,,hard-hearted
242,Zähigkeit,1,s,NaN,,,hard-hearted
243,zielorientiert,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive
244,zielstrebig,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive


In [12]:
#use split_form function to split alternatives into list
df_male_ct["Alternatives_split_role"] = df_male_ct["Alternatives-German Input fields job tite | Role | Requirements"].map(split_form)

In [13]:
df_male_ct["Alternatives_split_company"] = df_male_ct["Alternatives-German input fields Company | Company Culture | Goodies"].map(split_form)

In [14]:
#check
df_male_ct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 9 columns):
MaleCodedWords-German                                                   245 non-null object
Is active                                                               245 non-null int64
Wordtype                                                                245 non-null object
Forms                                                                   198 non-null object
Alternatives-German Input fields job tite | Role | Requirements         245 non-null object
Alternatives-German input fields Company | Company Culture | Goodies    245 non-null object
Subcategory: competitive | Hard-hearted | Rough | Corporate             240 non-null object
Alternatives_split_role                                                 245 non-null object
Alternatives_split_company                                              245 non-null object
dtypes: int64(1), object(8)
memory usage: 17.4+ KB


In [ ]:
#df_male_ct_new.drop(columns = ["test"], axis=0, inplace = True)

In [15]:
#check end of the file
df_male_ct.tail()

,MaleCodedWords-German,Is active,Wordtype,Forms,Alternatives-German Input fields job tite | Role | Requirements,Alternatives-German input fields Company | Company Culture | Goodies,Subcategory: competitive | Hard-hearted | Rough | Corporate,Alternatives_split_role,Alternatives_split_company
240,Wettkämpfe,1,s,NaN,,,competitive,[],[]
241,zäh,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,,,hard-hearted,[],[]
242,Zähigkeit,1,s,NaN,,,hard-hearted,[],[]
243,zielorientiert,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive,"[Meilensteine erreichen wollen, strebsam, flei...","[Meilensteine erreichen wollen, strebsam, flei..."
244,zielstrebig,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive,"[Meilensteine erreichen wollen, strebsam, flei...","[Meilensteine erreichen wollen, strebsam, flei..."


In [16]:
#Check if all rules are active
df_male_ct[df_male_ct["Is active"]!=1.0]

,MaleCodedWords-German,Is active,Wordtype,Forms,Alternatives-German Input fields job tite | Role | Requirements,Alternatives-German input fields Company | Company Culture | Goodies,Subcategory: competitive | Hard-hearted | Rough | Corporate,Alternatives_split_role,Alternatives_split_company


In [17]:
#save to csv file after preprocessing
df_male_ct.to_csv("MaleCodedTerms_DE.csv", index=False)

In [21]:
#upload file to check if it uploaded correctlz after the preprocessing
df_test = pd.read_csv("MaleCodedTerms_DE.csv")

In [22]:
df_test.tail()

,MaleCodedWords-German,Is active,Wordtype,Forms,Alternatives-German Input fields job tite | Role | Requirements,Alternatives-German input fields Company | Company Culture | Goodies,Subcategory: competitive | Hard-hearted | Rough | Corporate,Alternatives_split_role,Alternatives_split_company
240,Wettkämpfe,1,s,NaN,NaN,NaN,competitive,[''],['']
241,zäh,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,NaN,NaN,hard-hearted,[''],['']
242,Zähigkeit,1,s,NaN,NaN,NaN,hard-hearted,[''],['']
243,zielorientiert,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive,"['Meilensteine erreichen wollen', 'strebsam', ...","['Meilensteine erreichen wollen', 'strebsam', ..."
244,zielstrebig,1,a,(e|er|en|em|es|ere|erer|eren|erem|eres|erere|e...,Meilensteine erreichen wollen | strebsam | fle...,Meilensteine erreichen wollen | strebsam | fle...,competitive,"['Meilensteine erreichen wollen', 'strebsam', ...","['Meilensteine erreichen wollen', 'strebsam', ..."
